## Import packages

In [ ]:
import sys
sys.path.append('..')

import json
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from tensorflow.keras.models import load_model
from src import create_data_generators
from src import (
    evaluate_model, plot_confusion_matrix, plot_roc_curve,
    classification_report_dict
)
from sklearn.metrics import roc_curve, auc

plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams.update({
    'figure.facecolor': 'white',
    'axes.facecolor': 'white',
    'axes.labelcolor': 'black',
    'xtick.color': 'black',
    'ytick.color': 'black',
    'text.color': 'black',
    'legend.facecolor': 'white',
    'legend.edgecolor': 'black'
})

## Load test data (two sizes needed)

In [ ]:

data_dir = '../data/chest_xray'

# 224x224 for baseline and ResNet50
_, _, test_gen_224 = create_data_generators(
    data_dir, target_size=(224, 224), batch_size=32, augment=False
)

# 150x150 for VGG16
_, _, test_gen_150 = create_data_generators(
    data_dir, target_size=(150, 150), batch_size=32, augment=False
)


## Load all three models

In [ ]:
baseline_model = load_model('../models/baseline_model.keras')
resnet_model = load_model('../models/transfer_model.keras')
vgg16_model = load_model('../models/vgg16_150.keras')

print("All three models loaded successfully")

## Evaluate baseline model

In [ ]:
print("BASELINE CNN EVALUATION")
print("=" * 50)
baseline_results = evaluate_model(baseline_model, test_gen_224)


## Baseline confusion matrix

In [ ]:
plot_confusion_matrix(
    baseline_results['y_true'], baseline_results['y_pred'],
    baseline_results['class_names'],
    title="Baseline CNN - Confusion Matrix"
)

## Baseline classification report

In [ ]:
baseline_report = classification_report_dict(
    baseline_results['y_true'], baseline_results['y_pred'],
    baseline_results['class_names']
)

## Evaluate ResNet50 model

In [ ]:
print("\nTRANSFER LEARNING (ResNet50) EVALUATION")
print("=" * 50)
resnet_results = evaluate_model(resnet_model, test_gen_224)

## ResNet50 confusion matrix

In [ ]:
plot_confusion_matrix(
    resnet_results['y_true'], resnet_results['y_pred'],
    resnet_results['class_names'],
    title="ResNet50 - Confusion Matrix"
)

## ResNet50 classification report

In [ ]:
resnet_report = classification_report_dict(
    resnet_results['y_true'], resnet_results['y_pred'],
    resnet_results['class_names']
)

## Evaluate VGG16 model

In [ ]:
print("\nTRANSFER LEARNING (VGG16 150x150) EVALUATION")
print("=" * 50)
vgg16_results = evaluate_model(vgg16_model, test_gen_150)


## VGG16 confusion matrix

In [ ]:
plot_confusion_matrix(
    vgg16_results['y_true'], vgg16_results['y_pred'],
    vgg16_results['class_names'],
    title="VGG16 150x150 - Confusion Matrix"
)

## VGG16 classification report

In [ ]:
vgg16_report = classification_report_dict(
    vgg16_results['y_true'], vgg16_results['y_pred'],
    vgg16_results['class_names']
)

## ROC curves — all three models

In [ ]:
fig, ax = plt.subplots(figsize=(8, 6))

fpr_b, tpr_b, _ = roc_curve(baseline_results['y_true'], baseline_results['y_prob'])
auc_b = auc(fpr_b, tpr_b)
ax.plot(fpr_b, tpr_b, color='coral', linewidth=2, label=f'Baseline (AUC = {auc_b:.4f})')

fpr_r, tpr_r, _ = roc_curve(resnet_results['y_true'], resnet_results['y_prob'])
auc_r = auc(fpr_r, tpr_r)
ax.plot(fpr_r, tpr_r, color='steelblue', linewidth=2, label=f'ResNet50 (AUC = {auc_r:.4f})')

fpr_v, tpr_v, _ = roc_curve(vgg16_results['y_true'], vgg16_results['y_prob'])
auc_v = auc(fpr_v, tpr_v)
ax.plot(fpr_v, tpr_v, color='green', linewidth=2, label=f'VGG16 (AUC = {auc_v:.4f})')

ax.plot([0, 1], [0, 1], color='gray', linestyle='--', label='Random (AUC = 0.5)')

ax.set_xlabel('False Positive Rate', fontsize=12)
ax.set_ylabel('True Positive Rate', fontsize=12)
ax.set_title('ROC Curve Comparison — All Models', fontsize=14)
ax.legend(fontsize=11)
plt.tight_layout()
plt.show()

## Metrics comparison bar chart

In [ ]:
metrics = ['accuracy', 'precision', 'recall', 'f1']
baseline_vals = [baseline_results[m] for m in metrics]
resnet_vals = [resnet_results[m] for m in metrics]
vgg16_vals = [vgg16_results[m] for m in metrics]

x = np.arange(len(metrics))
width = 0.25

fig, ax = plt.subplots(figsize=(12, 6))
bars1 = ax.bar(x - width, baseline_vals, width, label='Baseline CNN', color='coral')
bars2 = ax.bar(x, resnet_vals, width, label='ResNet50', color='steelblue')
bars3 = ax.bar(x + width, vgg16_vals, width, label='VGG16', color='green')

ax.set_ylabel('Score')
ax.set_title('Model Comparison — All Metrics')
ax.set_xticks(x)
ax.set_xticklabels([m.capitalize() for m in metrics])
ax.legend()
ax.set_ylim(0, 1.1)

for bars in [bars1, bars2, bars3]:
    for bar in bars:
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
                f'{bar.get_height():.3f}', ha='center', fontsize=8)

plt.tight_layout()
plt.show()


## Detailed comparison table

In [ ]:
print("\nDETAILED MODEL COMPARISON")
print("=" * 75)
print(f"{'Metric':<12} {'Baseline':<12} {'ResNet50':<12} {'VGG16':<12} {'Winner':<15}")
print("-" * 75)

comparisons = {
    'Accuracy': (baseline_results['accuracy'], resnet_results['accuracy'], vgg16_results['accuracy']),
    'Precision': (baseline_results['precision'], resnet_results['precision'], vgg16_results['precision']),
    'Recall': (baseline_results['recall'], resnet_results['recall'], vgg16_results['recall']),
    'F1-Score': (baseline_results['f1'], resnet_results['f1'], vgg16_results['f1']),
    'AUC': (auc_b, auc_r, auc_v)
}

names = ['Baseline', 'ResNet50', 'VGG16']
for metric, vals in comparisons.items():
    winner = names[np.argmax(vals)]
    print(f"{metric:<12} {vals[0]:<12.4f} {vals[1]:<12.4f} {vals[2]:<12.4f} {winner:<15}")
print("=" * 75)

## Error analysis — misclassified images (best model: VGG16)

In [ ]:
test_gen_150.reset()
misclassified_indices = np.where(vgg16_results['y_true'] != vgg16_results['y_pred'])[0]

print(f"\nMisclassified images (VGG16): {len(misclassified_indices)} out of {len(vgg16_results['y_true'])}")
print(f"Error rate: {len(misclassified_indices) / len(vgg16_results['y_true']) * 100:.2f}%")

if len(misclassified_indices) > 0:
    test_gen_150.reset()
    all_images = []
    all_labels = []
    for i in range(len(test_gen_150)):
        batch_imgs, batch_labels = test_gen_150[i]
        all_images.append(batch_imgs)
        all_labels.append(batch_labels)
    all_images = np.concatenate(all_images)
    all_labels = np.concatenate(all_labels)

    n_show = min(8, len(misclassified_indices))
    fig, axes = plt.subplots(2, 4, figsize=(16, 8))
    fig.suptitle('Misclassified Images (VGG16 — Best Model)', fontsize=14)

    class_names = vgg16_results['class_names']
    for i, ax in enumerate(axes.flatten()[:n_show]):
        idx = misclassified_indices[i]
        ax.imshow(all_images[idx])
        true_label = class_names[int(vgg16_results['y_true'][idx])]
        pred_label = class_names[int(vgg16_results['y_pred'][idx])]
        conf = vgg16_results['y_prob'][idx]
        ax.set_title(f'True: {true_label}\nPred: {pred_label} ({conf:.2f})', fontsize=9)
        ax.axis('off')

    plt.tight_layout()
    plt.show()

## Summary

In [ ]:
print("=" * 50)
print("EVALUATION SUMMARY")
print("=" * 50)
print(f"\nBaseline CNN:")
print(f"  Accuracy: {baseline_results['accuracy'] * 100:.2f}%")
print(f"  AUC: {auc_b:.4f}")
print(f"\nResNet50 Transfer:")
print(f"  Accuracy: {resnet_results['accuracy'] * 100:.2f}%")
print(f"  AUC: {auc_r:.4f}")
print(f"\nVGG16 Transfer (150x150):")
print(f"  Accuracy: {vgg16_results['accuracy'] * 100:.2f}%")
print(f"  AUC: {auc_v:.4f}")
print(f"\nBest model: VGG16 150x150")
print(f"Improvement over baseline: {(vgg16_results['accuracy'] - baseline_results['accuracy']) * 100:+.2f}%")
print(f"Improvement over ResNet50: {(vgg16_results['accuracy'] - resnet_results['accuracy']) * 100:+.2f}%")
print(f"Misclassified: {len(misclassified_indices)} / {len(vgg16_results['y_true'])}")
print("=" * 50)
print("\nNext: 06_demo.ipynb")